# Pixel–Motif–Graph Hướng 1 — registered CRS PublicTest stage

Source-locked inference/evaluation wrapper for Issue #84 / Draft PR #85. This notebook performs no fitting and must be launched exactly once only after separate reviewer authorization. PrivateTest is not an input.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-composition-crs'
SOURCE_SHA = '175320c4a7215976891ba7d56c0c88bab168e873'
EXPECTED_PUBLIC_SHA256 = '412036d077c6ec203047b2935ab14bc858d8136ee26e8db3e23023f1fc9dee08'
EXPECTED_DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
EXPECTED_TRAIN_MODEL_SHA256 = '77b8a41d4a7de79b2216b4b9c7ad2d19e326cac25a46ec2a7a3dcaaa1f95607a'
FER_ROOT = Path('/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split')
PUBLIC_CSV = FER_ROOT / 'val.csv'
DICTIONARY_NPZ = Path('/kaggle/input/datasets/nuyntai/pgm-e01-v533-dictionary-crs/e01_dictionary.npz')
TRAIN_MODEL_NPZ = Path('/kaggle/input/datasets/nuyntai/pgm-crs-train-v2-artifacts/crs_train_model.npz')
OUTPUT_DIR = Path('/kaggle/working/outputs/pixel_relational_composition_crs_public')
PACKAGE_RELATIVE = Path('research/pixel_relational_motif_e0')
RUN_TESTS = True
RUN_PUBLIC_STAGE = True


In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'FER2013_Graph_CRS_PUBLIC'
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_SHA: raise RuntimeError(f'source lock mismatch: {head} != {SOURCE_SHA}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
subprocess.run(['git','-C',str(PROJECT),'diff','--cached','--quiet'], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents: raise RuntimeError(f'import isolation violation: {imported}')
print('Public scientific source lock PASS:', head)


In [ ]:
if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    result = subprocess.run([sys.executable,'-m','pytest',str(PACKAGE/'tests'),'-q'], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0: raise RuntimeError(f'pytest failed: {result.returncode}')
    print('Full package pytest PASS before any Public input access')


In [ ]:
import numpy as np, scipy, sklearn
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'sklearn': sklearn.__version__,
    'source_sha': SOURCE_SHA,
}
print('Environment:', json.dumps(environment, indent=2))
print('Working disk:', shutil.disk_usage('/kaggle/working'))


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

locked_inputs = {
    PUBLIC_CSV: EXPECTED_PUBLIC_SHA256,
    DICTIONARY_NPZ: EXPECTED_DICTIONARY_SHA256,
    TRAIN_MODEL_NPZ: EXPECTED_TRAIN_MODEL_SHA256,
}
for path, expected in locked_inputs.items():
    if not path.is_file(): raise FileNotFoundError(path)
    observed = sha256(path)
    if observed != expected: raise RuntimeError(f'input SHA mismatch for {path}: {observed}')
    print('Input lock PASS:', path, observed)


In [ ]:
import contextlib, threading, time
@contextlib.contextmanager
def heartbeat(label, interval_seconds=180):
    stop = threading.Event(); started = time.time()
    def worker():
        while not stop.wait(interval_seconds):
            print(f'[heartbeat] {label}: {(time.time()-started)/60:.1f} min', flush=True)
    thread = threading.Thread(target=worker, daemon=True); thread.start()
    try: yield
    finally:
        stop.set(); thread.join(timeout=1)
        print(f'[heartbeat] {label}: complete {(time.time()-started)/60:.1f} min', flush=True)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['CRS_PUBLIC_SOURCE_SHA'] = SOURCE_SHA
os.environ['CRS_EXECUTION_ACCOUNT'] = 'nuyntai'
os.environ['CRS_EXECUTION_KERNEL'] = 'nuyntai/pgm-crs-public-issue-84'
ENTRY = PROJECT / 'notebooks' / 'crs-public-entry.py'
if RUN_PUBLIC_STAGE:
    command = [sys.executable, str(ENTRY), '--public-csv', str(PUBLIC_CSV), '--dictionary-npz', str(DICTIONARY_NPZ), '--train-model-npz', str(TRAIN_MODEL_NPZ), '--output-dir', str(OUTPUT_DIR)]
    with heartbeat('registered CRS PublicTest inference/evaluation'):
        subprocess.run(command, check=True, env=os.environ.copy())


In [ ]:
required_outputs = [
    OUTPUT_DIR / 'crs_public_predictions.npz',
    OUTPUT_DIR / 'crs_public_summary.json',
    OUTPUT_DIR / 'crs_public_execution_manifest.json',
]
if not all(path.is_file() for path in required_outputs): raise RuntimeError('missing registered Public output')
summary = json.loads(required_outputs[1].read_text(encoding='utf-8'))
if summary.get('private_test_accessed') is not False: raise RuntimeError('PrivateTest isolation flag failed')
print(json.dumps(summary, indent=2, sort_keys=True))
print('Registered Public stage complete; PrivateTest remains sealed.')
